<a href="https://colab.research.google.com/github/eunmindow-star/chaeeunmin/blob/main/code/11_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
import pandas as pd
import numpy as np

drive.mount('/content/drive')
df = pd.read_sas('/content/drive/MyDrive/hdac_data/kyrbs2025.sas7bdat')

# 야식 섭취 관련 변수명 찾기 (보통 F_SNK 또는 F_NIGHT 등)
night_cols = [c for c in df.columns if 'F_' in c.upper() or 'SNK' in c.upper() or 'NIGHT' in c.upper()]
print("식생활/야식 관련 후보 변수:", night_cols)

Mounted at /content/drive
식생활/야식 관련 후보 변수: ['F_BR', 'F_FRUIT', 'F_CAFF_A', 'F_SWD_A', 'F_ZERO', 'F_FASTFOOD', 'F_VEG_A', 'F_MD', 'F_NE', 'F_WAT', 'E_FM_F_1', 'E_FM_SF_2', 'E_FM_GF_5', 'E_BORN_F_E']


In [3]:
# F_NE 변수의 기초 통계 및 빈도 분석
if 'F_NE' in df.columns:
    print("--- F_NE 변수 정보 ---")
    print(df['F_NE'].describe())
    print("\n--- 빈도수 분포 ---")
    print(df['F_NE'].value_counts(dropna=False).sort_index())
else:
    print("F_NE 변수를 찾을 수 없습니다. 다른 후보 변수를 탐색해야 합니다.")

--- F_NE 변수 정보 ---
count    54170.000000
mean         2.386099
std          1.656658
min          1.000000
25%          1.000000
50%          2.000000
75%          3.000000
max          8.000000
Name: F_NE, dtype: float64

--- 빈도수 분포 ---
F_NE
1.0    22989
2.0    10656
3.0     8981
4.0     5684
5.0     2573
6.0     1647
7.0      592
8.0     1048
Name: count, dtype: int64


In [8]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

# ==========================================
# 0. 변수명 정의
# ==========================================
selected_inc = 'E_SES'      # 주관적 경제수준 변수 확인됨
selected_slp = 'M_SLP_EN'   # 수면 충족감 주관적 평가 변수 확인됨

# ==========================================
# 1. 변수 전처리 (전체 54,170명 표본설계 구조 유지!)
# ==========================================
# 고등학생 도메인 지시변수 (자료를 자르지 않고 도메인으로 지정)
df['domain_hs'] = df['GRADE'].isin([4, 5, 6]).astype(int)

# 노출 (X): 아침식사 결식 (1: 주 5회 이상 결식[코드 1~3], 0: 비결식[코드 4~8])
df['bf_skip'] = np.where(df['F_BR'].isin([1, 2, 3]), 1,
                         np.where(df['F_BR'].notna(), 0, np.nan))

# 결과 (Y): 비만 유병 여부 (1: BMI >= 25, 0: BMI < 25)
df['bmi'] = df['WT'] / ((df['HT'] / 100) ** 2)
df['obesity'] = np.where(df['bmi'] >= 25, 1,
                         np.where(df['bmi'].notna(), 0, np.nan))

# 교란변수 (Confounders): 성별(SEX), 학년(GRADE), 경제상태(E_SES)
# - 성별: 1=남(0), 2=여(1)
df['female'] = np.where(df['SEX'] == 2, 1, np.where(df['SEX'] == 1, 0, np.nan))
# - 학년: 4=고1, 5=고2, 6=고3 (범주형 C(GRADE)로 투입)
# - 경제상태: 1~2=상(1), 3=중(2), 4~5=하(3), 9=무응답(NaN)
df['inc_grp'] = np.select(
    [df[selected_inc].isin([1, 2]), df[selected_inc] == 3, df[selected_inc].isin([4, 5])],
    [1, 2, 3], default=np.nan
)

# 매개 후보 (Mediators - Model 3 과보정 비교용): 수면(M_SLP_EN), 신체활동(PA_TOT), 야식(F_NE)
# - 수면충족감: 1~2=충분(1), 3=보통(2), 4~5=불충분(3), 9=무응답(NaN)
df['slp_grp'] = np.select(
    [df[selected_slp].isin([1, 2]), df[selected_slp] == 3, df[selected_slp].isin([4, 5])],
    [1, 2, 3], default=np.nan
)
# - 신체활동(PA_TOT): 1~3=주 0~2일 비실천(0), 4~8=주 3일 이상 실천(1), 9=무응답(NaN)
df['pa_act'] = np.where(df['PA_TOT'].isin([4, 5, 6, 7, 8]), 1,
                        np.where(df['PA_TOT'].isin([1, 2, 3]), 0, np.nan))
# - 야식 섭취(F_NE): 1=안 먹음(0), 2~8=주 1회 이상 섭취(1) (또는 주 3회 이상 등 기준 가능, 여기선 주 3회 이상[코드 4~8] vs 주 0~2회[코드 1~3])
df['night_eat'] = np.where(df['F_NE'].isin([4, 5, 6, 7, 8]), 1,
                           np.where(df['F_NE'].isin([1, 2, 3]), 0, np.nan))

# ==========================================
# 2. 복합표본 도메인 로지스틱 회귀분석 함수
# ==========================================
def fit_domain_survey_logit(data, formula, domain_col='domain_hs', weight_col='W', strata_col='STRATA', cluster_col='CLUSTER'):
    # 모형에 쓰이는 변수들 추출
    model_vars = [v.strip() for v in formula.replace('~', '+').replace('C(', '').replace(')', '').split('+')]

    # 유효 관측치 여부 (도메인 내 + 모형 변수 결측 없음)
    valid_in_domain = (data[domain_col] == 1) & data[model_vars].notna().all(axis=1)
    n_domain = int(valid_in_domain.sum())

    # 도메인 분석용 가중치
    w_domain = np.where(valid_in_domain, data[weight_col], 1e-12)

    # 결측치로 행이 탈락하지 않도록 도메인 밖 행의 임시 결측 채우기
    temp_df = data.copy()
    for v in model_vars:
        temp_df[v] = temp_df[v].fillna(temp_df[v].mode()[0])

    # 복합표본 층화·집락 그룹 키
    psu_group = temp_df[strata_col].astype(str) + "_" + temp_df[cluster_col].astype(str)

    # GLM Binomial (가중치 적용 + 집락-층화 기반 샌드위치 표준오차)
    model = smf.glm(
        formula=formula,
        data=temp_df,
        family=sm.families.Binomial(),
        var_weights=w_domain
    ).fit(cov_type='cluster', cov_kwds={'groups': psu_group})

    coef = model.params['bf_skip']
    se = model.bse['bf_skip']
    pval = model.pvalues['bf_skip']
    or_val = np.exp(coef)
    ci_low = np.exp(coef - 1.96 * se)
    ci_high = np.exp(coef + 1.96 * se)

    return {
        'n': n_domain,
        'OR': or_val,
        'CI_low': ci_low,
        'CI_high': ci_high,
        'SE': se,
        'p_value': pval,
        'formatted': f"{or_val:.2f} ({ci_low:.2f} - {ci_high:.2f})"
    }

# ==========================================
# 3. 3단계 모형 적합 및 결과 비교 출력
# ==========================================
f_m1 = "obesity ~ bf_skip"
f_m2 = "obesity ~ bf_skip + female + C(GRADE) + C(inc_grp)"
f_m3 = "obesity ~ bf_skip + female + C(GRADE) + C(inc_grp) + C(slp_grp) + pa_act + night_eat"

res_m1 = fit_domain_survey_logit(df, f_m1)
res_m2 = fit_domain_survey_logit(df, f_m2)
res_m3 = fit_domain_survey_logit(df, f_m3)

results_df = pd.DataFrame([
    {'모형 (Model)': 'Model 1 (무보정)', '보정 변수': '없음 (Crude)', '유효 표본 수(n)': f"{res_m1['n']:,}", 'OR (95% CI)': res_m1['formatted'], 'p-value': f"{res_m1['p_value']:.4f}"},
    {'모형 (Model)': 'Model 2 (주 모형: 교란보정)', '보정 변수': '성별, 학년, 경제상태', '유효 표본 수(n)': f"{res_m2['n']:,}", 'OR (95% CI)': res_m2['formatted'], 'p-value': f"{res_m2['p_value']:.4f}"},
    {'모형 (Model)': 'Model 3 (과보정: 매개포함)', '보정 변수': 'Model 2 + 수면, 신체활동, 야식', '유효 표본 수(n)': f"{res_m3['n']:,}", 'OR (95% CI)': res_m3['formatted'], 'p-value': f"{res_m3['p_value']:.4f}"}
])

print("=== [11주차] 고등학생 도메인 복합표본 로지스틱 회귀분석 결과 ===")
display(results_df)

=== [11주차] 고등학생 도메인 복합표본 로지스틱 회귀분석 결과 ===


,모형 (Model),보정 변수,유효 표본 수(n),OR (95% CI),p-value
0,Model 1 (무보정),없음 (Crude),"25,077",1.07 (1.00 - 1.15),0.0579
1,Model 2 (주 모형: 교란보정),"성별, 학년, 경제상태","25,070",1.09 (1.01 - 1.17),0.0215
2,Model 3 (과보정: 매개포함),"Model 2 + 수면, 신체활동, 야식","25,070",1.10 (1.02 - 1.18),0.0127


### 데이터셋 내 경제 상태 관련 변수명 검색
`PR_INC`, `PR_INC_A` 대신 실제 데이터셋에 어떤 경제수준 관련 변수명이 사용되었는지 확인합니다.

In [6]:
# INC(Income) 또는 PR(Personal/Perceived)이 들어가는 변수 목록 출력
inc_related = [c for c in df.columns if 'INC' in c.upper()]
pr_related = [c for c in df.columns if 'PR_' in c.upper()]

print("1. 'INC' 포함 컬럼:", inc_related)
print("2. 'PR_' 포함 컬럼:", pr_related)

1. 'INC' 포함 컬럼: []
2. 'PR_' 포함 컬럼: ['PR_HT', 'PR_BI', 'PR_OH']


### 데이터셋 전체 컬럼명 탐색
변수명 구조를 직접 확인하기 위해 전체 컬럼 리스트를 정렬하여 확인합니다.

In [7]:
# 경제상태 후보일 수 있는 'DEC' 변수가 있는지 및 전체 컬럼 출력
dec_cols = [c for c in df.columns if 'DEC' in c.upper()]
print("1. 'DEC' 포함 컬럼 (경제수준 후보):", dec_cols)
print("\n2. 전체 컬럼 목록:")
print(sorted(df.columns.tolist()))

1. 'DEC' 포함 컬럼 (경제수준 후보): []

2. 전체 컬럼 목록:
['AC_ACCESS', 'AC_AMNT', 'AC_DAYS', 'AC_DRUNK', 'AC_FAGE', 'AC_LT', 'AGE', 'AGE_M', 'A_FM', 'CITY', 'CLUSTER', 'CTYPE', 'CTYPE_SD', 'DR_HAB_PUR', 'E_BORN_F', 'E_BORN_F_E', 'E_BORN_M', 'E_BORN_M_E', 'E_EDU_F', 'E_EDU_M', 'E_FM_F_1', 'E_FM_GF_5', 'E_FM_GM_6', 'E_FM_M_3', 'E_FM_NO_9', 'E_FM_OBS_7', 'E_FM_SF_2', 'E_FM_SM_4', 'E_FM_YBS_8', 'E_KRN_F', 'E_KRN_M', 'E_RES', 'E_SES', 'E_S_RCRD', 'F_BR', 'F_CAFF_A', 'F_FASTFOOD', 'F_FRUIT', 'F_MD', 'F_NE', 'F_SWD_A', 'F_VEG_A', 'F_WAT', 'F_ZERO', 'GRADE', 'GROUP', 'HT', 'HW_SPGO_H', 'HW_SPML_H', 'HW_SPML_S', 'HW_SPRM_H', 'HW_SPRM_S', 'INT_SPWD', 'INT_SPWD_TM', 'INT_SPWK', 'INT_SPWK_TM', 'I_SB_BK', 'I_SB_EX', 'I_SB_FR', 'I_SCH_TRT', 'MH', 'M_SAD', 'M_SLP_EN', 'M_SLP_HR', 'M_SLP_HR_K', 'M_SLP_MM', 'M_SLP_MM_K', 'M_STR', 'M_SUI_ATT', 'M_SUI_CON', 'M_SUI_PLN', 'M_WK_HR', 'M_WK_HR_K', 'M_WK_MM', 'M_WK_MM_K', 'OBS', 'O_BR_FQ', 'O_BR_S', 'O_BR_SLP', 'O_DC_FQ', 'O_OCP1', 'O_OCP2', 'O_OCP3', 'O_OCP4', 'O_OCP5', '